# Modelagem — Score de detração pós-viagem

Este notebook treina e avalia o modelo que estima, para cada jornada encerrada, a probabilidade de o Cliente se tornar Detrator. Ele começa onde o `pre-processamento.ipynb` termina: consome a base analítica já integrada e não repete nenhuma etapa de limpeza ou integração.

Princípios adotados:

- consumir a base analítica pronta, sem reprocessar as fontes brutas;
- importar a lógica de seleção, divisão e transformação de `scripts/preprocessamento_nps.py`, nunca reescrevê-la aqui;
- ajustar imputação, codificação e escala somente no treino;
- manter o conjunto de teste intocado até a etapa de avaliação;
- versionar o notebook sem saídas de execução e sem nenhum dado do parceiro.

O notebook é preenchido por etapas, cada uma com card próprio no board:

| Seção | Conteúdo | Card |
|---|---|---|
| 1 | Contrato de entrada e carga da matriz | #100 |
| 2 | Baselines de referência | #101 |
| 3 | Validação cruzada por Cliente | #102 |
| 4 | Primeiro modelo candidato | #103 |
| 5 | Ajuste de hiperparâmetros | #104 |
| 6 | Métricas de ordenação | #105 |
| 7 | Limiar operacional e matriz de confusão | #106 |
| 8 | Curvas ROC e precisão contra cobertura | #107 |
| 9 | Tabela comparativa | #108 |


## 1. Contrato de entrada

Esta seção fixa o que a etapa de carga e organização dos dados precisa entregar para que a modelagem rode. O contrato existe para que as duas frentes avancem em paralelo: enquanto a organização não estiver pronta, basta que o arquivo abaixo exista com estas colunas para que todo o restante do notebook funcione sem alteração.

**Arquivo esperado:** `data/processed/base_analitica.parquet`, produzido pela seção 13 do `notebooks/pre-processamento.ipynb`. No Colab, o mesmo arquivo pode vir do Drive; a busca de caminho está na célula seguinte e não exige edição de código.

**Colunas obrigatórias.** A base precisa carregar três grupos:

| Grupo | Colunas | Para que serve |
|---|---|---|
| Alvo | `DETRATOR` | variável a ser predita, binária |
| Chaves de partição | `DATA_STD_CONVERTIDA`, `ID_GOLDENRECORD` | corte temporal e agrupamento por Cliente |
| Features | as 11 de `FEATURE_SET_V1` | matriz do modelo |

As 11 features são a allowlist do score pós-viagem definida no módulo de pré-processamento e não são reescritas aqui: `TIER_VIAGEM`, `VOO_TIPO`, `TIPO_ENTRETENIMENTO`, `CANAL_COMPRA`, `SEGMENTO`, `ESTATISTICA_ATRASOSAIDA`, `ATRASO_CHEGADA`, `CANCELAMENTO_VOO`, `ANTECEDENCIA_CANCELAMENTO`, `TEMPO_VOO`, `N_TRECHOS`. A lista canônica é o `FEATURE_SET_V1`, que o módulo também expõe sob o nome antigo `FEATURES_SCORE_POS_VIAGEM`, e é ele que a célula 1.1 importa. O `QTDE_VIAGENS_12M` ocupava a última posição até o `feat(#149)` e está suspenso pela seção 4.3.2.3 da documentação, enquanto não houver forma auditável de recalcular o histórico com corte em `t_score`. Qualquer mudança nessa lista é decisão de modelagem e acontece no módulo, não no notebook.

**Partições.** A divisão não é aleatória. `split.dividir`, chamada dentro de `preparar_matriz` (seção 1.4), aplica corte temporal e por Cliente em três vias — treino, validação e teste —, retirando de cada conjunto anterior todo Cliente que reaparece num conjunto mais recente, porque a base tem dependência intracliente e um Cliente presente em dois lados inflaria o desempenho estimado. A função devolve as três partições como DataFrames completos, e é pelo índice de cada uma que features, alvo e grupos são referenciados no restante do notebook, nunca por posição.

**O que não pode entrar na matriz do modelo.** Nenhuma coluna iniciada por `NPS_` ou `SUB_`. São respostas coletadas no mesmo instrumento que origina o alvo, e o contrato temporal da seção 4.2.3 da documentação as exclui do modelo.

A proibição vale para a matriz, e não para a base analítica. A base carrega essas colunas por construção: `DETRATOR` é derivado de `NPS_PRINCIPAL` dentro da própria `preparar_base_analitica`, e a persistência em Parquet da seção 13 do pré-processamento não descarta coluna alguma. Exigir uma base sem elas faria o notebook falhar sobre uma entrada correta. Por isso as duas verificações têm papéis distintos, e isso é deliberado:

| Célula | Papel | Interrompe a execução |
|---|---|---|
| 1.3 | diagnóstico da base analítica: confere colunas obrigatórias e features da allowlist, e apenas relata quantas colunas de pesquisa vieram junto | sim para coluna obrigatória ausente, não para coluna de pesquisa presente |
| 1.4 | contrato da matriz do modelo: confere que o conjunto de features é exatamente a allowlist e que nenhuma coluna de pesquisa entrou, na entrada e na saída do pré-processador | sim |


### 1.1. Dependências e módulo de pré-processamento

In [ ]:
!pip install -q pandas numpy scikit-learn pyarrow matplotlib seaborn

import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Inclui a pasta do módulo tanto no projeto local quanto no Colab. O break evita
# que duas cópias de preprocessamento_nps.py fiquem no caminho ao mesmo tempo,
# o que tornaria ambíguo qual delas o import resolve.
for pasta_codigo in [Path("scripts"), Path("../scripts"), Path("/content/scripts")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break

import preprocessamento_nps
importlib.reload(preprocessamento_nps)

from preprocessamento_nps import (
    FEATURES_SCORE_POS_VIAGEM,
    validar_parquet,
)

SEMENTE = 42
np.random.seed(SEMENTE)

# O contrato da seção 1 ancora em 11 features. Uma mudança silenciosa na
# allowlist do módulo precisa falhar aqui, e não apenas aparecer no print.
QUANTIDADE_FEATURES_ESPERADA = 11
if len(FEATURES_SCORE_POS_VIAGEM) != QUANTIDADE_FEATURES_ESPERADA:
    raise AssertionError(
        f"A allowlist mudou: {len(FEATURES_SCORE_POS_VIAGEM)} features, "
        f"esperadas {QUANTIDADE_FEATURES_ESPERADA}. "
        "Atualize o contrato de entrada da seção 1 antes de seguir."
    )

print("features do score pós-viagem:", len(FEATURES_SCORE_POS_VIAGEM))


### 1.2. Localização e carga da base analítica

A busca cobre o projeto local e o Colab, com e sem Drive montado. Se nenhum caminho existir, a célula falha com a instrução do que fazer, em vez de seguir com uma base vazia.


In [ ]:
# No Colab, monte o Drive antes de rodar esta célula:
#   from google.colab import drive; drive.mount('/content/drive')

CAMINHOS_POSSIVEIS = [
    Path("data/processed/base_analitica.parquet"),
    Path("../data/processed/base_analitica.parquet"),
    Path("/content/data/processed/base_analitica.parquet"),
    Path("/content/drive/MyDrive/projeto-azul/data/processed/base_analitica.parquet"),
]

caminho_base = next((caminho for caminho in CAMINHOS_POSSIVEIS if caminho.exists()), None)

if caminho_base is None:
    raise FileNotFoundError(
        "Base analítica não encontrada. Gere-a com a seção 13 de notebooks/pre-processamento.ipynb "
        "ou monte o Drive e ajuste CAMINHOS_POSSIVEIS. Caminhos testados: "
        + ", ".join(str(caminho) for caminho in CAMINHOS_POSSIVEIS)
    )

validar_parquet(caminho_base)
df = pd.read_parquet(caminho_base)

print("base analítica:", caminho_base)
print("linhas:", f"{len(df):,}".replace(",", "."), "| colunas:", df.shape[1])


### 1.3. Diagnóstico da base analítica

A célula confere o que a base precisa ter e interrompe a execução se faltar coluna obrigatória ou feature da allowlist. Falhar aqui é barato; descobrir a ausência depois de treinar não é.

A contagem de colunas `NPS_` e `SUB_` é diagnóstico e não motivo de parada, porque a base analítica carrega essas colunas por construção, como explicado na seção 1. O que não pode acontecer é elas alcançarem a matriz do modelo, e é a seção 1.4 que bloqueia isso.


In [ ]:
COLUNAS_OBRIGATORIAS = {"DETRATOR", "DATA_STD_CONVERTIDA", "ID_GOLDENRECORD"}

faltando = COLUNAS_OBRIGATORIAS - set(df.columns)
if faltando:
    raise KeyError(f"A base não atende ao contrato de entrada. Colunas ausentes: {sorted(faltando)}")

features_ausentes = [coluna for coluna in FEATURES_SCORE_POS_VIAGEM if coluna not in df.columns]
if features_ausentes:
    raise KeyError(f"Features da allowlist ausentes na base: {features_ausentes}")

# Diagnóstico, e não trava: a base analítica carrega as respostas da pesquisa
# por construção, já que DETRATOR é derivado de NPS_PRINCIPAL. O bloqueio
# efetivo acontece sobre a matriz do modelo, na seção 1.4.
colunas_pesquisa = [coluna for coluna in df.columns if coluna.startswith(("NPS_", "SUB_"))]
print("colunas de pesquisa na base analítica:", len(colunas_pesquisa), "(diagnóstico, não bloqueia)")
print("prevalência de Detrator:", f"{df['DETRATOR'].mean():.2%}")


### 1.4. Matriz de modelagem e contrato da matriz

`preparar_matriz`, de `src/matriz.py`, executa numa chamada as três etapas que precisam
acontecer nesta ordem: particionar por data e por Cliente, selecionar a allowlist de features e
ajustar o pré-processador **apenas no treino**. Inverter as duas últimas seria inofensivo;
ajustar o pré-processador antes de particionar não, porque a mediana do treino passaria a
carregar informação de 2026 e a métrica de teste mediria um modelo que já viu o período que
deveria prever.

As datas de corte não são escolhidas aqui. Elas vêm do registro de decisão em
`documents/extras/politica-de-particionamento-temporal.md`, que é a fonte única: repeti-las como
constante do notebook criaria uma segunda verdade que poderia divergir da documentada.

`sem_data="treino"` envia ao treino qualquer linha sem data — hoje cerca de 120 mil na base
analítica, remanescentes de respostas antigas que a Azul não datou. Enviá-las ao treino em vez
de descartá-las evita perder um quinto da base; a garantia de que isso não corrompe o corte
temporal vem de `verificar_anterioridade_sem_data`, que prova pela ordem de registro
(`RESPONDENT_ID`) que essas linhas antecedem o período datado. `split.dividir` chama essa
verificação por conta própria sempre que existir alguma linha sem data, interrompendo a
execução se a prova deixar de valer.

O contrato da matriz é cobrado nas duas pontas do pré-processador. Na entrada, pega uma allowlist
alterada; na saída, uma transformação que reintroduza uma coluna de pesquisa sob outro nome.
Qualquer uma das duas interrompe a execução.


In [ ]:
for pasta_codigo in [Path("src"), Path("../src"), Path("/content/src")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break

import matriz
importlib.reload(matriz)
from matriz import preparar_matriz, resumo_da_matriz

# Datas do registro de decisão do #127, não escolhidas aqui.
CORTE_VALIDACAO = "2025-07-01"
CORTE_TESTE = "2026-01-01"

preparo = preparar_matriz(
    df,
    corte_validacao=CORTE_VALIDACAO,
    corte_teste=CORTE_TESTE,
    sem_data="treino",
)

x_treino, x_validacao, x_teste = (preparo["x"][k] for k in ("treino", "validacao", "teste"))
y_treino, y_validacao, y_teste = (preparo["y"][k] for k in ("treino", "validacao", "teste"))
grupos_treino = preparo["grupos"]["treino"]
preprocessador = preparo["preprocessador"]
x_treino_transformado = preparo["matrizes"]["treino"]
x_validacao_transformado = preparo["matrizes"]["validacao"]
x_teste_transformado = preparo["matrizes"]["teste"]
numericas = preparo["metadados"]["numericas"]
categoricas = preparo["metadados"]["categoricas"]
metadados_divisao = preparo["metadados"]

display(resumo_da_matriz(preparo))

anterioridade = metadados_divisao["anterioridade_sem_data"]
# verificar_anterioridade_sem_data devolve dois formatos: um quando ha linha
# sem data para verificar, outro (sem as chaves de correlacao) quando o bloco
# esta vazio, que e o caso normal na base atual.
if anterioridade["linhas_sem_data"]:
    print("linhas sem data enviadas ao treino:", f"{anterioridade['linhas_sem_data']:,}".replace(",", "."))
    print("  correlação de Spearman entre ordem de registro e data:", anterioridade["correlacao_spearman"])
    print("  maior id sem data:", anterioridade["maior_id_sem_data"],
          "< menor id datado:", anterioridade["menor_id_datado"])
else:
    print("nenhuma linha sem data na base: a verificação de anterioridade não se aplica")
print("removidas por Cliente recorrente:",
      f"{metadados_divisao['linhas_removidas_por_recorrencia']:,}".replace(",", "."))
print("features:", len(metadados_divisao["features_usadas"]),
      "| numéricas:", len(numericas), "| categóricas:", len(categoricas),
      "| colunas da matriz:", metadados_divisao["colunas_da_matriz"])


### 1.5. Composição das partições (CR05 do #127/#128)

Esta seção não reparticiona: `split.resumo` recebe `preparo["particoes"]` e
`preparo["metadados"]`, já calculados na 1.4, e devolve n, intervalo de datas,
Clientes e percentual do total por partição. Chamar `split.dividir` de novo aqui
geraria a mesma divisão duas vezes no mesmo notebook, com o risco de as duas
chamadas divergirem se uma delas for editada e a outra não.


In [ ]:
from split import resumo

tabela_particoes = resumo(preparo["particoes"], metadados=preparo["metadados"])
print(tabela_particoes)


### 1.6. Auditoria de representatividade dos conjuntos (#130)

Antes de treinar, esta seção confere se treino, validação e teste são comparáveis entre si e
com a base completa — em taxa de detração e na distribuição de quatro variáveis-chave. Sem essa
checagem, uma queda de métrica no teste não se pode atribuir ao modelo: pode ser só uma diferença
de composição entre os períodos.

As quatro variáveis auditadas são `TIER_VIAGEM`, `VOO_TIPO`, `FAIXA_ATRASO` (a mesma taxonomia de
`src/clean.py`, reaproveitada aqui em vez de reinventada) e `SEGMENTO`. O card original citava
"região ou rota" como quarta variável, mas a base analítica não tem nenhuma coluna de região,
rota, origem ou destino. O candidato mais próximo, `VOO_INTERNACIONAL`, tem um único valor em
toda a base (`DOMESTIC`, 484.915 de 484.915 linhas) — sem variância, não audita nada — por isso
`SEGMENTO` entra no lugar.


In [ ]:
from clean import ORD_ATRASO, faixa_atraso

CATEGORICAS_AUDITORIA = {
    "TIER_VIAGEM": sorted(df["TIER_VIAGEM"].dropna().unique()),
    "VOO_TIPO": sorted(df["VOO_TIPO"].dropna().unique()),
    "SEGMENTO": sorted(df["SEGMENTO"].dropna().unique()),
}
LIMIAR_DESVIO_PP = 2.0


def perfil_particao(sub):
    faixa = faixa_atraso(sub["ESTATISTICA_ATRASOSAIDA"])
    linha = {"n": len(sub), "taxa_detracao_pct": round(sub["DETRATOR"].mean() * 100, 2)}
    for categoria in ORD_ATRASO:
        linha[f"FAIXA_ATRASO={categoria}"] = round((faixa == categoria).mean() * 100, 2)
    for coluna, valores in CATEGORICAS_AUDITORIA.items():
        for valor in valores:
            linha[f"{coluna}={valor}"] = round((sub[coluna] == valor).mean() * 100, 2)
    return linha


def contagem_particao(sub):
    """Contagem bruta por categoria, para o teste de suporte no treino.

    Um desvio em pontos percentuais pode ficar abaixo do limiar mesmo quando o
    treino tem suporte zero para uma categoria que aparece em validação ou
    teste: a proporção de uma categoria rara muda pouco entre conjuntos
    grandes, mas "pouco" e "zero" nao sao a mesma coisa para o OneHotEncoder,
    que so aprende categorias vistas no ajuste.
    """
    faixa = faixa_atraso(sub["ESTATISTICA_ATRASOSAIDA"])
    contagem = {f"FAIXA_ATRASO={c}": int((faixa == c).sum()) for c in ORD_ATRASO}
    for coluna, valores in CATEGORICAS_AUDITORIA.items():
        for valor in valores:
            contagem[f"{coluna}={valor}"] = int((sub[coluna] == valor).sum())
    return contagem


particoes_com_base = {**preparo["particoes"], "base completa": df}
tabela_auditoria = pd.DataFrame(
    {nome: perfil_particao(sub) for nome, sub in particoes_com_base.items()}
).T

tabela_auditoria["desvio_taxa_detracao_pp"] = (
    tabela_auditoria["taxa_detracao_pct"] - tabela_auditoria.loc["base completa", "taxa_detracao_pct"]
).round(2)

display(tabela_auditoria)

colunas_composicao = [
    c for c in tabela_auditoria.columns if c not in ("n", "taxa_detracao_pct", "desvio_taxa_detracao_pp")
]
desvios_composicao = tabela_auditoria.loc[
    tabela_auditoria.index != "base completa", colunas_composicao
].subtract(tabela_auditoria.loc["base completa", colunas_composicao], axis=1).round(2)

desvios_taxa = tabela_auditoria.loc[tabela_auditoria.index != "base completa", "desvio_taxa_detracao_pp"]
relevantes_taxa = desvios_taxa[desvios_taxa.abs() > LIMIAR_DESVIO_PP]
relevantes_composicao = desvios_composicao[desvios_composicao.abs() > LIMIAR_DESVIO_PP].stack().dropna()

print(f"desvios acima de {LIMIAR_DESVIO_PP} p.p. na taxa de detração:")
print("  nenhum" if relevantes_taxa.empty else relevantes_taxa.to_string())
print(f"desvios acima de {LIMIAR_DESVIO_PP} p.p. na composição das variáveis-chave:")
print("  nenhum" if relevantes_composicao.empty else relevantes_composicao.to_string())

# Suporte no treino: categoria com contagem zero no treino, mas presente em
# validação ou teste, vira vetor zero no OneHotEncoder (handle_unknown=
# "ignore", src/matriz.py) e some da informação disponível ao modelo — um
# problema de ausência de suporte, não de diferença de proporção, que o
# desvio em p.p. acima não pega.
contagens = pd.DataFrame(
    {nome: contagem_particao(sub) for nome, sub in preparo["particoes"].items()}
).T
sem_suporte_no_treino = contagens.columns[
    (contagens.loc["treino"] == 0) & (contagens.loc[["validacao", "teste"]].sum() > 0)
]
print(f"categorias sem suporte no treino, presentes em validação ou teste:")
if len(sem_suporte_no_treino) == 0:
    print("  nenhuma")
else:
    print(contagens.loc[["treino", "validacao", "teste"], sem_suporte_no_treino].to_string())

**Conclusão da auditoria (#130).** A taxa de detração é estável entre os três conjuntos —
20,43% no treino, 21,60% na validação e 20,41% no teste, contra 20,44% na base completa — com
desvio máximo de 1,16 p.p. (validação), abaixo do limiar de 2 p.p. adotado aqui.

A composição das variáveis-chave tem onze ocorrências acima do limiar, que colapsam em oito
desvios distintos: `VOO_TIPO=DIRETO` espelha `VOO_TIPO=CONEXÃO` em cada partição (mesma
magnitude, sinal oposto), então as duas contam como um único desvio de composição. Os cinco que
não envolvem `VOO_TIPO` são moderados, todos abaixo de 2,7 p.p., sem padrão monotônico ao longo
do tempo: `TIER_VIAGEM=AZUL FIDELIDADE` (+2,70 p.p.) e `TIER_VIAGEM=DIAMANTE` (-2,67 p.p.) no
treino; `FAIXA_ATRASO=a. Sem Atraso` (+2,27 p.p.) e `FAIXA_ATRASO=b. 15m - 60m` (-2,65 p.p.) na
validação; `TIER_VIAGEM=TOPAZIO` (-2,37 p.p.) no teste.

`VOO_TIPO` é a exceção relevante, e não só no teste: a participação de voos com conexão cai de
forma monotônica ao longo do tempo em todas as três partições — +2,59 p.p. no treino (32,08%
contra 29,49% na base completa), -4,28 p.p. na validação (25,21%) e -8,14 p.p. no teste (21,35%).
É deriva real entre períodos, não ruído de amostragem, e precisa ser reportada na Seção 4.3
(#116).

Duas categorias de `TIER_VIAGEM` não têm suporte algum no treino: `AZUL ONE` (189 linhas, 0,35%
do teste) e `DIAMANTE UNIQUE` (1.164 linhas, 2,18% do teste) não aparecem nem no treino nem na
validação, só no teste. Em pontos percentuais o desvio fica abaixo do limiar e passaria
despercebido, mas o problema não é de proporção: é ausência de suporte. O `OneHotEncoder`
(`handle_unknown="ignore"`, `src/matriz.py`) não interrompe a execução — transforma essas 1.353
linhas (2,5% do teste) em vetor zero para a dimensão de tier, e o modelo fica sem essa
informação justamente nas linhas mais recentes da base.

**A divisão é aceitável como está.** A taxa de detração, a variável mais próxima do que o modelo
precisa acertar, não mostra deriva relevante, e os cinco desvios de composição fora de `VOO_TIPO`
são moderados. Duas limitações reais precisam ser declaradas na Seção 4.3, não omitidas: a
deriva monotônica de `VOO_TIPO` nas três partições, e a ausência de suporte no treino para
`AZUL ONE` e `DIAMANTE UNIQUE`, que reduz a informação de tier disponível ao modelo em 2,5% do
teste.

### 1.7. Tabela e figura de composição dos conjuntos (#132)

Reaproveita `tabela_particoes`, já calculada na 1.5, para gerar a tabela de composição (n,
período coberto, taxa de detração e percentual do total) e a figura da linha do tempo do
particionamento. Os dois artefatos vão para `documents/extras/composicao-dos-conjuntos.md` e
`assets/`, prontos para a redação da Seção 4.3 no #116 — este card não escreve a prosa da seção.


In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

from graficos import PALETA_AZUL, aplicar_tema

RAIZ_REPO = caminho_base.parents[2]

# ---- tabela de composição, direto de tabela_particoes (sem digitação manual) ----
tabela_composicao = tabela_particoes[["n", "data_inicio", "data_fim", "taxa_alvo_pct", "pct_do_total"]].copy()
tabela_composicao.columns = ["n", "início", "fim", "taxa de detração (%)", "% do total"]

NOMES_EXIBICAO = {"treino": "Treino", "validacao": "Validação", "teste": "Teste"}

linhas_md = ["| Conjunto | n | Período | Taxa de detração | % do total |",
             "|---|---|---|---|---|"]
for particao, linha in tabela_composicao.iterrows():
    periodo = f"{linha['início']} a {linha['fim']}"
    n_fmt = f"{int(linha['n']):,}".replace(",", ".")
    linhas_md.append(
        f"| {NOMES_EXIBICAO[particao]} | {n_fmt} | {periodo} | "
        f"{linha['taxa de detração (%)']:.2f}% | {linha['% do total']:.1f}% |"
    )
tabela_composicao_md = "\n".join(linhas_md)
print(tabela_composicao_md)

# ---- figura da linha do tempo do particionamento ----
aplicar_tema()

data_col = pd.to_datetime(df["DATA_STD"], errors="coerce")
inicio_base, fim_base = data_col.min(), data_col.max()
corte_val, corte_teste = pd.Timestamp(CORTE_VALIDACAO), pd.Timestamp(CORTE_TESTE)

fig, ax = plt.subplots(figsize=(10, 3.6))
cores = {"treino": PALETA_AZUL[0], "validação": PALETA_AZUL[1], "teste": PALETA_AZUL[2]}
for nome, ini, fimseg in (
    ("treino", inicio_base, corte_val),
    ("validação", corte_val, corte_teste),
    ("teste", corte_teste, fim_base),
):
    ax.axvspan(ini, fimseg, color=cores[nome], alpha=0.85, label=nome.capitalize())

for corte, rotulo in ((corte_val, "corte validação"), (corte_teste, "corte teste")):
    ax.axvline(corte, color="#0A2A6B", linestyle="--", linewidth=1.2)
    ax.annotate(f"{rotulo}\n{corte.date()}", xy=(corte, 0.5), xycoords=("data", "axes fraction"),
                ha="center", va="center", fontsize=8.5, color="white", fontweight="bold",
                bbox={"boxstyle": "round,pad=0.3", "facecolor": "#0A2A6B", "edgecolor": "none", "alpha": 0.85})

ax.set_yticks([])
ax.set_xlim(inicio_base, fim_base)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b/%Y"))
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_title("Linha do tempo do particionamento temporal", pad=14)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.55), ncol=3, frameon=False)

notas_rodape = []
cobertura_pct = tabela_particoes["pct_do_total"].sum()
notas_rodape.append(
    f"os três conjuntos cobrem {cobertura_pct:.1f}% da base — o restante "
    f"({100 - cobertura_pct:.1f}%) foi removido por Cliente recorrente ou por não ter "
    "Cliente, sem posição temporal própria, e não aparece no eixo"
)

n_sem_data = metadados_divisao["linhas_sem_data"]
if n_sem_data:
    notas_rodape.append(
        f"{n_sem_data:,}".replace(",", ".") + " linhas sem data, enviadas ao treino (fora do eixo)"
    )

fig.text(0.02, 0.01, "\n".join(notas_rodape), fontsize=8, color="#333")

fig.subplots_adjust(bottom=0.46, top=0.88)
caminho_figura = RAIZ_REPO / "assets" / "linha-tempo-particionamento.png"
caminho_figura.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(caminho_figura)
plt.close(fig)
print("figura salva em", caminho_figura)


## 2. Baselines de referência

Um número de desempenho só significa alguma coisa quando existe um piso para compará-lo. Esta seção estabelece dois pisos, e todo modelo apresentado depois é julgado contra eles.

O primeiro é a **regra da classe majoritária**, que não aprende nada e existe para expor o quanto a acurácia engana numa base em que cerca de um em cada cinco Clientes é Detrator. O segundo é uma **regressão logística**, modelo linear e interpretável, que mede quanto do sinal já é capturado sem recorrer a interação entre variáveis. É contra esse segundo piso que a escolha de um método baseado em árvores, na seção 4, precisa se justificar.

Os dois são ajustados apenas na partição de treino e medidos sobre a mesma partição de teste que o modelo candidato usará, sem o que a comparação não seria válida.

**Escopo desta seção.** A acurácia e a precisão média aparecem aqui como piso de comparação, não como a resposta ao item (c) da Seção 4.3. A escolha e a explicação das três métricas oficiais do modelo são daquele item e ficam na seção 6 deste notebook.

### 2.1. Prevalência e função de avaliação

A prevalência da classe Detrator é o número que dá sentido a todo o resto: é ela que a precisão média de um modelo sem informação reproduz, e é dela que a acurácia do baseline trivial se aproxima por complemento.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, average_precision_score,
                             roc_auc_score)

print(f"prevalência de Detrator no treino: {y_treino.mean():.2%}")
print(f"prevalência de Detrator no teste:  {y_teste.mean():.2%}")


def avaliar_no_teste(nome, modelo):
    """Mede um modelo já treinado sobre a partição de teste temporal.

    A acurácia entra ao lado da precisão média de propósito. É a distância
    entre as duas que evidencia o efeito do desbalanceamento, e é isso que
    justifica não apresentar acurácia como medida de qualidade neste problema.

    O ROC-AUC entra porque a tabela comparativa da seção 9 cobra as três
    métricas oficiais definidas na seção 4.1.3, e ele é a única das três que
    já pode ser medida aqui: depende apenas da ordenação do score. Revocação
    em Detrator e precisão dependem do ponto de corte, que só é decidido na
    seção 7, e medi-las agora seria fixar um limiar arbitrário antes da
    seção que existe para escolhê-lo.
    """
    probabilidades = modelo.predict_proba(x_teste_transformado)[:, 1]
    return {
        "modelo": nome,
        "acuracia": accuracy_score(y_teste, modelo.predict(x_teste_transformado)),
        "precisao_media": average_precision_score(y_teste, probabilidades),
        "roc_auc": roc_auc_score(y_teste, probabilidades),
    }

### 2.2. Baseline trivial: regra da classe majoritária

O classificador responde sempre a classe mais frequente do treino, que é a de não Detrator. Ele não olha nenhuma feature.

O resultado esperado é uma acurácia próxima de quatro quintos, acompanhada de uma precisão média próxima da prevalência. É essa contradição que interessa: um modelo que erra todos os Detratores da base ainda assim exibe acurácia alta. A probabilidade que ele emite é degenerada, constante para toda linha, e é por isso que a precisão média desaba para o piso. Pela mesma razão o ROC-AUC fica em 0,5: sem variação no score não existe ordenação, e 0,5 é exatamente o valor de quem ordena ao acaso.

In [ ]:
trivial = DummyClassifier(strategy="most_frequent", random_state=SEMENTE)
trivial.fit(x_treino_transformado, y_treino)

resultado_trivial = avaliar_no_teste("classe majoritária", trivial)
print(f"acurácia:       {resultado_trivial['acuracia']:.4f}")
print(f"precisão média: {resultado_trivial['precisao_media']:.4f}")
print(f"ROC-AUC:        {resultado_trivial['roc_auc']:.4f}")

### 2.3. Baseline linear: regressão logística

O segundo piso já aprende, mas apenas efeitos aditivos: cada variável contribui com um peso fixo, sem que o modelo represente combinações entre elas.

O parâmetro de balanceamento de classe é declarado explicitamente. Sem ele a logística se ajusta à classe majoritária e passa a quase não prever Detrator, o que a tornaria indistinguível do baseline trivial e inútil como piso. A matriz chega aqui já escalonada pelo pré-processador da seção 1.4, condição para que os coeficientes sejam comparáveis entre variáveis de unidades diferentes.

In [ ]:
from sklearn.linear_model import LogisticRegression

logistica = LogisticRegression(
    class_weight="balanced",   # declarado: sem isso o modelo colapsa na classe majoritária
    # O padrão de max_iter é 1000 e não basta aqui: o lbfgs ainda não
    # convergiu nesse limite, e um modelo interrompido antes da convergência
    # não serve como piso, porque os coeficientes ainda estariam mudando.
    # 5000 dá folga sem custo relevante nesta matriz.
    max_iter=5000,
    random_state=SEMENTE,
)
logistica.fit(x_treino_transformado, y_treino)

resultado_logistica = avaliar_no_teste("regressão logística", logistica)
print(f"acurácia:       {resultado_logistica['acuracia']:.4f}")
print(f"precisão média: {resultado_logistica['precisao_media']:.4f}")
print(f"ROC-AUC:        {resultado_logistica['roc_auc']:.4f}")

### 2.4. Coeficientes da regressão logística

Os coeficientes ficam acessíveis no notebook para leitura posterior. Eles indicam a direção com que cada variável empurra o risco de detração, e voltam a ser usados na discussão de resultados, quando for preciso dizer se o modelo candidato encontrou algo que o piso linear não via.

In [ ]:
coeficientes_logistica = pd.DataFrame({
    "feature": preprocessador.get_feature_names_out(),
    "coeficiente": logistica.coef_[0],
}).sort_values("coeficiente", key=abs, ascending=False, ignore_index=True)

print("intercepto:", float(logistica.intercept_[0]))
coeficientes_logistica.head(15)

### 2.5. Resultado consolidado dos dois pisos

A tabela abaixo é o piso de comparação que as seções seguintes consomem. A leitura que importa: o baseline trivial exibe a maior acurácia da tabela e a menor precisão média, e a regressão logística inverte essa relação. O ROC-AUC completa a leitura: o trivial fica no 0,5 de quem não ordena nada, e é a distância da logística para esse 0,5 que mede o sinal já capturado por um modelo puramente aditivo. Qualquer modelo apresentado depois precisa superar a segunda linha na precisão média para que seu ganho seja atribuível ao modelo, e não ao desbalanceamento da base.

In [ ]:
tabela_baselines = pd.DataFrame([resultado_trivial, resultado_logistica])

ganho = resultado_logistica["precisao_media"] - resultado_trivial["precisao_media"]
print(f"ganho da logística sobre o trivial na precisão média: {ganho:+.4f}")
if ganho <= 0:
    print("ATENÇÃO: a logística não superou o piso trivial; revise o balanceamento antes de seguir.")

tabela_baselines

## 3. Validação cruzada por Cliente

`GroupKFold` de 5 folds dentro do treino, agrupando por `ID_GOLDENRECORD`.

**Por que agrupada, e não aleatória.** As respostas de um mesmo Cliente não são independentes: a
hipótese 4 da seção 4.2.3 mediu que quem detratou uma vez volta a detratar com chance 4,75 vezes
maior, e que o efeito sobrevive ao controle por atraso e cancelamento. Num `KFold` comum, duas
respostas da mesma pessoa cairiam em folds diferentes, e o modelo acertaria a segunda por
reconhecer a primeira. A média dos cinco folds subiria sem que nada tivesse melhorado, e o
hiperparâmetro escolhido em cima dela seria escolhido sobre um número inflado.

É a mesma razão que governa a divisão treino/teste em `split.dividir`, aplicada um nível abaixo.
Aqui ela é ainda mais necessária: o corte temporal não protege nada dentro do treino, porque os
folds não são temporais.

**Por que folds e não um terceiro bloco temporal.** A partição já reserva os meses finais para o
teste; recortar mais um bloco encolheria o treino justamente na faixa em que o histórico de
Cliente começa a existir. Com os folds, cada linha do treino serve de validação exatamente uma
vez, sem sair do ajuste.

A conferência abaixo não é decorativa: `conferir_folds` interrompe a execução se um Cliente
aparecer na validação de dois folds, ou no ajuste e na validação do mesmo fold. O `GroupKFold`
garante as duas coisas por construção, mas a garantia depende de o vetor de grupos ser o certo, e
passar o vetor errado produz folds tecnicamente válidos e cientificamente errados.

In [ ]:
from validacao import (conferir_folds, criar_folds, metadados_da_divisao,
                       resumo_dos_folds)

# Os indices dos folds sao posicionais, entao valem igual para x_treino e para
# x_treino_transformado: preparar_matriz constroi os dois a partir das mesmas
# linhas, na mesma ordem.
folds = criar_folds(x_treino, grupos_treino)
conferir_folds(folds, grupos_treino)

print(resumo_dos_folds(folds, grupos_treino, y_treino))
print()
for chave, valor in metadados_da_divisao(preparo).items():
    print(f"{chave}: {valor}")

## 4. Primeiro modelo candidato

Classificador de gradient boosting sobre árvores, escolha justificada pela ausência de preditor dominante e pela interação entre fidelização e falha operacional documentadas na seção 4.2.1.

*A preencher no card #103.*


### 4.1. Hiperparâmetros declarados e treino do candidato

`HIPERPARAMETROS_CANDIDATO` declara todos os hiperparâmetros que importam, inclusive os que
coincidem com o padrão da biblioteca, porque padrão não declarado é decisão que ninguém tomou.
Dois deles merecem leitura antes de rodar:

- `early_stopping=False`. No padrão `"auto"` o `HistGradientBoostingClassifier` liga a parada
  antecipada sozinho acima de dez mil linhas e separa uma fatia **aleatória** do ajuste para
  medir. Essa fatia não respeita `ID_GOLDENRECORD`, então respostas do mesmo Cliente cairiam no
  ajuste e na medição interna, que é o vazamento que a seção 3 existe para evitar.
- `class_weight=None`, ao contrário da logística da seção 2.3. Lá a reponderação era necessária
  para o piso não colapsar na classe majoritária. Aqui as duas métricas usadas dependem apenas da
  ordenação do score, que a reponderação não melhora, e ela deslocaria a probabilidade predita
  para longe da frequência observada, o que estragaria o escore de Brier da seção 6 e o limiar
  por capacidade da seção 7.

A semente é a mesma `SEMENTE` da seção 1.1, e é ela que faz duas execuções seguidas devolverem o
mesmo número.


In [ ]:
import modelo
importlib.reload(modelo)
from modelo import (HIPERPARAMETROS_CANDIDATO, comparar_nos_folds,
                    conferir_ganho_sobre_os_pisos, criar_candidato)

for nome, valor in HIPERPARAMETROS_CANDIDATO.items():
    print(f"{nome}: {valor}")
print(f"random_state: {SEMENTE}")

# O candidato final vê a partição de treino inteira, com o pré-processador já
# ajustado na seção 1.4. É este objeto que as seções 5 a 8 consomem.
candidato = criar_candidato(semente=SEMENTE)
candidato.fit(x_treino_transformado, y_treino)

# `n_iter_` é igual a max_iter por construção, porque `early_stopping=False`
# desliga a parada antecipada. O print não mede nada que varie, serve de
# conferência da decisão: se a parada antecipada voltar a ligar sozinha, os dois
# números deixam de bater e isso aparece aqui.
print(f"\niterações de boosting: {candidato.n_iter_}"
      f" (max_iter declarado: {HIPERPARAMETROS_CANDIDATO['max_iter']})")


### 4.2. Desempenho na validação contra os dois pisos

A comparação acontece nos cinco folds da seção 3, e os três modelos passam pelos mesmos folds.
Duas decisões sustentam o número:

- **Fábrica, e não modelo pronto.** Cada fold recebe um estimador novo. Reaproveitar a instância
  já ajustada faria o segundo fold começar do ajuste do primeiro, e a média dos cinco mediria um
  modelo que já viu quase todo o treino.
- **Pré-processador reajustado dentro de cada fold.** O da seção 1.4 é ajustado na partição de
  treino inteira, o que é correto para o modelo final, mas usá-lo aqui deixaria a mediana da
  imputação e a escala carregarem informação das linhas que o fold usa como validação. O efeito
  seria pequeno e invisível, a pior combinação possível num número que decide hiperparâmetro na
  seção seguinte.

Os dois pisos são os mesmos da seção 2, com a mesma parametrização, para que a diferença medida
seja de algoritmo e não de configuração. A precisão média é a métrica da comparação porque
resume a curva de precisão contra revocação, que é a leitura adequada quando só um em cada cinco
Clientes é Detrator.

Esta é a célula mais cara do notebook: são quinze ajustes, cinco por algoritmo.


In [ ]:
from sklearn.base import clone

# As fábricas dos dois pisos clonam os objetos das células 2.2 e 2.3 em vez de
# repetir a configuração deles. `clone` devolve um estimador novo, ainda não
# ajustado, com os mesmos parâmetros, que é exatamente o que `avaliar_nos_folds`
# pede. Copiar a configuração aqui deixaria um ajuste na logística da seção 2
# fora da comparação por folds, e as duas medidas passariam a falar de modelos
# diferentes sem nada no caminho avisar.
fabricas = {
    "classe majoritária": lambda: clone(trivial),
    "regressão logística": lambda: clone(logistica),
    "gradient boosting": lambda: criar_candidato(semente=SEMENTE),
}

comparacao_validacao = comparar_nos_folds(
    fabricas, x_treino, y_treino, folds, preprocessador,
)

# CR02 do #103 como trava: sem ela, um candidato pior que o piso seguiria para o
# ajuste da seção 5 e só a tabela da seção 9 denunciaria o problema.
conferir_ganho_sobre_os_pisos(
    comparacao_validacao,
    candidato="gradient boosting",
    pisos=["classe majoritária", "regressão logística"],
)

display(comparacao_validacao)


**Leitura do resultado.** Medida na base analítica atual, a média dos cinco folds:

| Modelo | Precisão média | Desvio entre folds | ROC-AUC |
|---|---|---|---|
| classe majoritária | 0,2043 | 0,0011 | 0,5000 |
| regressão logística | 0,4766 | 0,0061 | 0,7337 |
| gradient boosting | 0,5023 | 0,0060 | 0,7424 |

O candidato supera os dois pisos, e o que sustenta a afirmação não é a média sozinha: ele fica à
frente da regressão logística **nos cinco folds, um a um**, com ganho de 0,026 na precisão média,
pouco mais de quatro vezes o desvio entre folds. Uma diferença menor que esse desvio não
autorizaria a conclusão, e é por isso que o desvio aparece na tabela.

O piso trivial se comporta como a seção 2.2 previu: precisão média de 0,2043, que reproduz a
prevalência de Detrator no treino, e ROC-AUC de 0,5, o valor de quem não ordena nada.

**O tamanho do ganho, dito com honestidade.** A distância entre o candidato e a logística é de
0,026, enquanto a distância entre a logística e o piso trivial é de 0,272. A maior parte do sinal
desta base já é linear, e a interação entre fidelização e falha operacional acrescenta uma fração
menor, consistente mas menor. Isso não desfaz a escolha do algoritmo, que segue justificada
porque o ganho é real e aparece em todos os folds, mas dimensiona o que ela entrega: a redação da
Seção 4.3 deve apresentar o boosting como um avanço medido, não como um salto. O ajuste de
hiperparâmetros da seção 5 é a próxima chance de mover esse número, e é ele que dirá se 0,026 é o
teto do método nesta base ou apenas o resultado da configuração inicial.

## 5. Ajuste de hiperparâmetros

Busca sobre taxa de aprendizado, profundidade e número de iterações, avaliada apenas nos folds de validação.

*A preencher no card #104.*


## 6. Métricas de ordenação do score

Precisão média, área sob a curva ROC e escore de Brier, calculadas sobre o conjunto de teste.

*A preencher no card #105.*


## 7. Limiar operacional e matriz de confusão

Ponto de corte derivado da capacidade diária de contato da equipe de Experiência do Cliente, e não fixado em 0,5 por omissão.

*A preencher no card #106.*


## 8. Curvas ROC e de precisão contra cobertura

Figuras salvas em `assets/`, no mesmo padrão das da seção 4.2.

*A preencher no card #107.*


## 9. Tabela comparativa

Baselines e modelo candidato nas mesmas métricas e sobre a mesma partição de teste.

*A preencher no card #108.*
